In [1]:
import pandas as pd
import re

In [23]:
base_path_data = "../../../data"
base_path_cv = base_path_data + "/processed/model_performance/cross_validation"
base_path_held_out = base_path_data + "/processed/model_performance/held_out"

# read metadata
metadata = pd.read_json(f"{base_path_data}/raw/metadata_ku.json", lines=True)
metadata_unique = metadata.groupby("new_sub_id").first().reset_index()

## Cross validation model performance

In [70]:
# function to extract subject ID from evaluation results filename
def extract_sub_id(filename):
    match = re.match(r"colon_(0*\d+)-", filename)
    if match:
        return f"sub{int(match.group(1)):03d}"
    return None

# read results
cv_results = {}
for fold_number in range(5):
    df = pd.read_csv(f"{base_path_cv}/results_fold_{fold_number}.csv")
    # group by filename and keep only the first row per file as we look at overall performance
    df = df.groupby("filename").first().reset_index()
    df["sub_id"] = df["filename"].apply(extract_sub_id)

    # merge evaluation results with metadata on subject ID (per scan)
    df = pd.merge(
        df,
        metadata_unique,
        left_on="sub_id",
        right_on="new_sub_id",
        how="left"
    )

    df["gender"] = df["gender"].apply(lambda x: x if x in ["M", "F"] else "U")
    grouped_by_gender =  df.groupby("gender")

    metrics = {
        "dice": {
            "count": grouped_by_gender["overall_dice_score"].count(),
            "mean": grouped_by_gender["overall_dice_score"].mean(),
            "std": grouped_by_gender["overall_dice_score"].std()
        },
        "hf95": {
            "count": grouped_by_gender["overall_hausdorff_distance_95th"].count(),
            "mean": grouped_by_gender["overall_hausdorff_distance_95th"].mean(),
            "std": grouped_by_gender["overall_hausdorff_distance_95th"].std()
        },
        "assd": {
            "count": grouped_by_gender["overall_average_symmetric_surface_distance"].count(),
            "mean": grouped_by_gender["overall_average_symmetric_surface_distance"].mean(),
            "std": grouped_by_gender["overall_average_symmetric_surface_distance"].std()
        },
    }
        
    cv_results[fold_number] = metrics

In [89]:
# print results
for fold in cv_results:
    print(f"------------------------ FOLD {fold} ------------------------")
    print(f"-------------- DICE --------------")
    print(pd.DataFrame(cv_results[fold]["dice"]))
    print(f"-------------- HF95 --------------")
    print(pd.DataFrame(cv_results[fold]["hf95"]))
    print(f"-------------- ASSD --------------")
    print(pd.DataFrame(cv_results[fold]["assd"]))


------------------------ FOLD 0 ------------------------
-------------- DICE --------------
        count      mean      std
gender                          
F          70  0.971808  0.01442
M          55  0.969084  0.02650
U          12  0.964214  0.02210
-------------- HF95 --------------
        count      mean       std
gender                           
F          70  2.424610  4.443671
M          55  2.596959  5.355975
U          12  4.256145  5.507208
-------------- ASSD --------------
        count      mean       std
gender                           
F          70  0.535022  0.362869
M          55  0.554227  0.408648
U          12  0.778512  0.550994
------------------------ FOLD 1 ------------------------
-------------- DICE --------------
        count      mean       std
gender                           
F          66  0.973294  0.011489
M          56  0.970655  0.013785
U          15  0.963584  0.026107
-------------- HF95 --------------
        count      mean        std
g

## Held out test set model performance

In [93]:
# function to extract subject ID from evaluation results filename
def extract_sub_id(filename):
    match = re.match(r"colon_(0*\d+)-", filename)
    if match:
        return f"sub{int(match.group(1)):03d}"
    return None

# read results
held_out_results = {}
file_paths = ["results_held_out_collapsed.csv", "results_held_out_noncollapsed.csv", "combined"]

for file_path in file_paths:
    if file_path == "combined":
        df1 = pd.read_csv(f"{base_path_held_out}/{file_paths[0]}")
        df2 = pd.read_csv(f"{base_path_held_out}/{file_paths[1]}")
        df = pd.concat([df1, df2])
    else:
        df = pd.read_csv(f"{base_path_held_out}/{file_path}")

    # group by filename and keep only the first row per file as we look at overall performance
    df = df.groupby("filename").first().reset_index()
    df["sub_id"] = df["filename"].apply(extract_sub_id)

    # merge evaluation results with metadata on subject ID (per scan)
    df = pd.merge(
        df,
        metadata_unique,
        left_on="sub_id",
        right_on="new_sub_id",
        how="left"
    )

    df["gender"] = df["gender"].apply(lambda x: x if x in ["M", "F"] else "U")
    grouped_by_gender =  df.groupby("gender")

    metrics = {
        "dice": {
            "count": grouped_by_gender["overall_dice_score"].count(),
            "mean": grouped_by_gender["overall_dice_score"].mean(),
            "std": grouped_by_gender["overall_dice_score"].std()
        },
        "hf95": {
            "count": grouped_by_gender["overall_hausdorff_distance_95th"].count(),
            "mean": grouped_by_gender["overall_hausdorff_distance_95th"].mean(),
            "std": grouped_by_gender["overall_hausdorff_distance_95th"].std()
        },
        "assd": {
            "count": grouped_by_gender["overall_average_symmetric_surface_distance"].count(),
            "mean": grouped_by_gender["overall_average_symmetric_surface_distance"].mean(),
            "std": grouped_by_gender["overall_average_symmetric_surface_distance"].std()
        },
    }
        
    held_out_results[file_path] = metrics

In [98]:
for results in held_out_results:
    print(f"------------------------ {results} ------------------------")
    print(f"-------------- DICE --------------")
    print(pd.DataFrame(held_out_results[results]["dice"]))
    print(f"-------------- HF95 --------------")
    print(pd.DataFrame(held_out_results[results]["hf95"]))
    print(f"-------------- ASSD --------------")
    print(pd.DataFrame(held_out_results[results]["assd"]))


------------------------ results_held_out_collapsed.csv ------------------------
-------------- DICE --------------
        count      mean       std
gender                           
F          26  0.972054  0.008802
M          12  0.974527  0.004285
-------------- HF95 --------------
        count      mean       std
gender                           
F          26  1.207219  0.494295
M          12  1.069036  0.161232
-------------- ASSD --------------
        count      mean       std
gender                           
F          26  0.452789  0.170509
M          12  0.418653  0.057460
------------------------ results_held_out_noncollapsed.csv ------------------------
-------------- DICE --------------
        count      mean       std
gender                           
F          14  0.970012  0.010681
M          16  0.970741  0.009986
U           2  0.965536  0.001763
-------------- HF95 --------------
        count      mean       std
gender                           
F          14 